# CurvLearn — Difficulty sweep: is curvature ever loss-relevant?

Design A found kappa moves/freezes but is **loss-neutral** on easy Dyck-3. Before scheduling
kappa (Design B) is worth anything, we need a task whose loss actually depends on kappa.

This sweep **fixes** kappa at a grid of values (no learnable-kappa dynamics to confound the
reading) and measures **close-bracket loss** at increasing hierarchical difficulty, over 3
seeds. Watch for a **hyperbolic dip** (close-loss lower at kappa<0) emerging on the hard task
while easy stays flat. Flat everywhere = tunable curvature does not help this family.

Set runtime to **GPU**. Results persist to Drive and the sweep is checkpointed/resumable.


## 1. GPU


In [ ]:
!nvidia-smi -L || echo 'no GPU (CPU is slow for 45 runs)'

## 2. Code + install (autoreload on, so pulls need no restart)


In [ ]:
%load_ext autoreload
%autoreload 2
import os
if not os.path.isdir('CurvLearn') and os.path.basename(os.getcwd()) != 'CurvLearn':
    !git clone https://github.com/TheFausap/CurvLearn.git
if os.path.basename(os.getcwd()) != 'CurvLearn':
    os.chdir('CurvLearn')
!pip -q install -e .
print('cwd:', os.getcwd())

## 3. Drive + output folder


In [ ]:
import os, torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/CurvLearn/results'
except ModuleNotFoundError:
    OUTDIR = 'results'
os.makedirs(OUTDIR, exist_ok=True)
print('device:', dev, '| results ->', OUTDIR)

## 4. Run the difficulty sweep
3 difficulties x 5 kappa x 3 seeds = 45 runs, ~40 min on an A100. Checkpointed to Drive after
every run; re-run this cell to resume. Lower `seeds` or `steps` for a faster first look.


In [ ]:
import json
from curvlearn.difficulty import run_difficulty, default_spec

spec = default_spec()
spec['train']['device'] = dev
ckpt = os.path.join(OUTDIR, 'difficulty_results.json')
out = run_difficulty(spec, checkpoint_path=ckpt, resume=True)
print('runs:', len(out['records']), '| saved ->', ckpt)

## 5. Plot: does a hyperbolic dip emerge?


In [ ]:
import json
from curvlearn import plot
from IPython.display import Image, display
out = json.load(open(os.path.join(OUTDIR, 'difficulty_results.json')))
png = os.path.join(OUTDIR, 'difficulty_sweep.png')
plot.difficulty_plot(out, png)
display(Image(png))
print('saved', png)

## 6. Read it
- **Right panel dives below 0 at kappa<0 for `hard` (but not `easy`)** -> curvature becomes
  loss-relevant as the hierarchy deepens: hyperbolic geometry buys capacity the flat model
  lacks. That is the green light for Design B (schedule kappa toward the useful curvature).
- **All lines stay flat** -> curvature is loss-neutral even on the hard task at this scale.
  Then the lever to try next is smaller `d_model` (tighter capacity) or a genuine
  tree-distance target, not scheduling kappa.
- **`hard` curve rises at kappa<0** -> hyperbolic actively hurts here; the task doesn't want it.
